# 03 — Feature engineering

Build leakage-safe predictors and 24-hour target vectors for the target station in the wide joined partitions.

**Inputs:** `data/processed/joined/all_stations_train.parquet`, `all_stations_test.parquet`
**Outputs:** `data/processed/joined/all_stations_{train,test}_features.parquet` and `all_stations_feature_metadata.json`

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.config import STATION_IDS, TARGET_STATION_ID, WEATHER_VARIABLES
from src.feature_engineering import (
    DEFAULT_FEATURE_CONFIG,
    build_feature_frame,
    extract_station_frame,
    feature_column_names,
    target_column_names,
    write_joined_feature_artifacts,
)

JOINED_DIR = Path("data/processed/joined")
PARTITIONS = ("train", "test")
ENGINEERED_STATION_IDS = (TARGET_STATION_ID,)
SOURCE_COLUMNS = ("water_level", "imputed", "station_id", *WEATHER_VARIABLES)
DERIVED_FEATURE_COLUMNS = tuple(
    column
    for column in feature_column_names(DEFAULT_FEATURE_CONFIG)
    if column not in SOURCE_COLUMNS
)
DERIVED_COLUMNS = (
    *DERIVED_FEATURE_COLUMNS,
    "target_valid",
    *target_column_names(DEFAULT_FEATURE_CONFIG),
)
OUTPUT_PATHS = {
    partition: JOINED_DIR / f"all_stations_{partition}_features.parquet"
    for partition in PARTITIONS
}

## Leakage contract

The wide train and sealed-test joined Parquets are read independently. Only the target station's prefixed source columns are extracted into a single-station frame before calling `build_feature_frame`; the beginning of the test partition cannot inherit train lookbacks. Upstream station columns remain the raw joined inputs.

Every predictor uses information available through issue time $t$: positive shifts create lags, and trailing windows include $t$, require a complete window, and propagate source missingness. Future weather is never used.

Targets are the target station's `water_level` at $t+1$ through $t+24$. `target_valid` is calculated independently inside the target station frame and requires the full future target window to be observed and non-imputed; otherwise all 24 target values are null. Original joined columns and the shared timestamp are preserved unchanged. Only the target prefix receives derived predictors, `target_valid`, and targets.

No rows are dropped from the joined outputs, no missing values are filled, no scaling is applied, and no `features_valid` shortcut is introduced. Upstream `station_id` values remain raw metadata and upstream raw measurements, flags, and weather remain available as predictors.

In [ ]:
def attach_station_features(
    joined: pd.DataFrame,
    station_id: str,
    available: pd.Series,
    station_features: pd.DataFrame,
) -> pd.DataFrame:
    """Add one station's derived columns while preserving the joined frame."""
    result = joined.copy()
    values_index = joined.index[available]
    prefix = f"{station_id}__"
    for column in DERIVED_COLUMNS:
        if column == "target_valid":
            values = pd.Series(pd.NA, index=joined.index, dtype="boolean")
        else:
            values = pd.Series(float("nan"), index=joined.index, dtype="float64")
        if not station_features.empty:
            values.loc[values_index] = station_features[column].to_numpy()
        result[f"{prefix}{column}"] = values
    return result


def build_joined_feature_frame(
    joined: pd.DataFrame, partition: str
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    """Engineer only the target station in one joined partition."""
    result = joined.copy()
    partition_summaries: list[dict[str, object]] = []
    station_id = TARGET_STATION_ID
    station = extract_station_frame(joined, station_id)
    available = joined[f"{station_id}__station_id"].notna()
    if station.empty:
        result = attach_station_features(
            result, station_id, available, station_features=station
        )
        target_valid_rows = 0
    else:
        station_features = build_feature_frame(
            station, station_id=station_id, config=DEFAULT_FEATURE_CONFIG
        )
        result = attach_station_features(
            result, station_id, available, station_features
        )
        target_valid_rows = int(station_features["target_valid"].sum())
    partition_summaries.append(
        {
            "partition": partition,
            "station_id": station_id,
            "usable_rows": int(available.sum()),
            "target_valid_rows": target_valid_rows,
        }
    )
    return result, partition_summaries


summaries: list[dict[str, object]] = []
partition_features: dict[str, pd.DataFrame] = {}
for partition in PARTITIONS:
    source_path = JOINED_DIR / f"all_stations_{partition}.parquet"
    joined_source = pd.read_parquet(source_path)
    joined_features, partition_summaries = build_joined_feature_frame(
        joined_source, partition
    )
    partition_features[partition] = joined_features
    for summary in partition_summaries:
        summaries.append(
            {
                **summary,
                "rows": len(joined_features),
                "columns": len(joined_features.columns),
                "path": str(OUTPUT_PATHS[partition]),
            }
        )

train_features = partition_features["train"]
test_features = partition_features["test"]

manifest = write_joined_feature_artifacts(
    train_features,
    test_features,
    station_ids=STATION_IDS,
    engineered_station_ids=ENGINEERED_STATION_IDS,
    train_source_path=JOINED_DIR / "all_stations_train.parquet",
    test_source_path=JOINED_DIR / "all_stations_test.parquet",
    output_dir=JOINED_DIR,
)

In [ ]:
artifact_summary = pd.DataFrame(summaries)
contract_summary = pd.DataFrame(
    {
        "contract": [
            "original joined columns",
            "engineered stations",
            "derived predictors for target station",
            "raw predictors per upstream station",
            "targets for target station",
            "forecast horizon (hours)",
        ],
        "count": [
            len(joined_source.columns),
            len(ENGINEERED_STATION_IDS),
            len(DERIVED_FEATURE_COLUMNS),
            len(SOURCE_COLUMNS) - 1,
            len(target_column_names(DEFAULT_FEATURE_CONFIG)),
            DEFAULT_FEATURE_CONFIG.horizon_hours,
        ],
    }
)
display(artifact_summary)
display(contract_summary)

Wide joined feature outputs were regenerated successfully. Source columns and timestamps are preserved, the target station is engineered independently in each partition, and upstream stations remain raw joined inputs.